# <font color="#418FDE" size="6.5" uppercase>**Imputation Targets**</font>

>Last update: 20260713.
    
By the end of this Lecture, you will be able to:
- Apply univariate, multivariate, and neighbor-based imputation to incomplete data. 
- Use missingness indicators and understand estimators with native missing-value support. 
- Transform classification and regression targets using scikit-learn target utilities. 


## **1. Imputation Methods**

### **1.1. Basic Value Imputation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_B/image_01_01.jpg?v=1783983480" width="250">



>* Fill blanks with simple fixed values
>* Fast, clear, and model-friendly baseline

>* Match imputation to variable type and distribution
>* Use unknown categories when missingness matters

>* Simple imputation can distort data patterns
>* Use as a baseline, not default



In [ ]:
#@title Python Code - Basic Value Imputation

# Demonstrate simple imputation for missing feature values.
# Compare mean, median, and most frequent replacements.
# Show how SimpleImputer fills a small table.

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
import sklearn

# Build a tiny dataset with numeric and categorical gaps.
data = pd.DataFrame(
    {
        "age": [29, np.nan, 41, 38, np.nan],
        "income": [52000, 61000, np.nan, 58000, 120000],
        "color": ["blue", "red", np.nan, "blue", np.nan],
    }
)

# Validate the table shape before transforming it.
if data.shape != (5, 3):
    raise ValueError("Expected five rows and three columns.")

# Use separate imputers for different feature meanings.
age_imputer = SimpleImputer(strategy="median")
income_imputer = SimpleImputer(strategy="mean")
color_imputer = SimpleImputer(strategy="most_frequent")

# Fit each imputer on observed values, then fill missing entries.
filled_data = data.copy()
filled_data[["age"]] = age_imputer.fit_transform(data[["age"]])
filled_data[["income"]] = income_imputer.fit_transform(data[["income"]])
filled_data[["color"]] = color_imputer.fit_transform(data[["color"]])

# Round numeric columns for beginner-friendly display.
filled_data["age"] = filled_data["age"].round(1)
filled_data["income"] = filled_data["income"].round(0).astype(int)

print("scikit-learn version:", sklearn.__version__)
print("Missing values before:", int(data.isna().sum().sum()))
print("Missing values after:", int(filled_data.isna().sum().sum()))
print("Imputed table:")
print(filled_data.to_string(index=False))



### **1.2. Iterative Imputation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_B/image_01_02.jpg?v=1783983481" width="250">



>* Predict missing values using other features
>* Best when variables share useful patterns

>* Repeatedly updates missing feature estimates
>* Uses other columns to predict gaps

>* Flexible models require careful judgment
>* Best with useful predictors and moderate missingness



In [ ]:
#@title Python Code - Iterative Imputation

# Demonstrate iterative imputation on correlated numeric features.
# Missing values are predicted from other columns.
# Compare rough gaps with model-based filled estimates.

import numpy as np
import pandas as pd
import sklearn
from sklearn.experimental import enable_iterative_imputer

from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge

# Create a small correlated dataset with deterministic randomness.
rng = np.random.default_rng(42)
study_hours = rng.normal(10, 2, 12)
attendance = 55 + study_hours * 3 + rng.normal(0, 3, 12)

# Grades depend on both study hours and attendance.
prior_grade = 40 + study_hours * 2 + attendance * 0.25
prior_grade = prior_grade + rng.normal(0, 2, 12)

# Put the features into a beginner-friendly table.
data = pd.DataFrame(
    {"study_hours": study_hours, "attendance": attendance, "prior_grade": prior_grade}
)

# Add missing values in more than one feature.
missing_data = data.copy()
missing_data.loc[[2, 7], "study_hours"] = np.nan
missing_data.loc[[4, 9], "attendance"] = np.nan

# Validate the table before fitting the imputer.
if missing_data.shape != (12, 3):
    raise ValueError("Expected a 12 row by 3 column teaching table.")

# IterativeImputer repeatedly predicts each incomplete column.
imputer = IterativeImputer(
    estimator=BayesianRidge(), max_iter=10, random_state=42, initial_strategy="mean"
)
filled_array = imputer.fit_transform(missing_data)

# Convert the filled array back to labeled columns.
filled_data = pd.DataFrame(filled_array, columns=missing_data.columns)
missing_locations = missing_data.isna()

# Show only the cells that were originally missing.
print("scikit-learn version:", sklearn.__version__)
print("Original missing cells filled by IterativeImputer:")

for row_index, column_name in zip(*np.where(missing_locations.to_numpy())):
    label = missing_data.columns[column_name]
    value = filled_data.iloc[row_index, column_name]
    print("row", row_index, label, "->", round(value, 2))



### **1.3. Nearest Neighbor Imputation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_B/image_01_03.jpg?v=1783983483" width="250">



>* Fill gaps using similar records
>* Local patterns guide context-aware estimates

>* Uses multiple variables for flexible imputation
>* Scale and encode features for fair similarity

>* Similarity quality and data completeness matter
>* Tune neighbors and evaluate downstream impact



In [ ]:
#@title Python Code - Nearest Neighbor Imputation

# Demonstrate nearest neighbor imputation on small tabular data.
# Similar rows help estimate each missing numeric value.
# Printed results compare original and imputed values.

import numpy as np
import pandas as pd
import sklearn
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

# Create a tiny housing dataset with related numeric features.
housing = pd.DataFrame(
    {
        "size_sqft": [900, 950, 1200, 1250, 1800, 1850],
        "bedrooms": [2, 2, 3, 3, 4, 4],
        "bathrooms": [1.0, np.nan, 2.0, np.nan, 3.0, 3.5],
        "price_k": [220, 230, 310, 320, 470, 490],
    }
)

# Validate the small table shape before transforming it.
if housing.shape != (6, 4):
    raise ValueError("Expected a 6 row by 4 column teaching table.")

# Scale features so square footage does not dominate distances.
scaler = StandardScaler()
scaled_values = scaler.fit_transform(housing)

# KNNImputer borrows values from the two most similar rows.
imputer = KNNImputer(n_neighbors=2)
imputed_scaled_values = imputer.fit_transform(scaled_values)

# Convert imputed scaled values back to the original units.
imputed_values = scaler.inverse_transform(imputed_scaled_values)
imputed_housing = pd.DataFrame(imputed_values, columns=housing.columns)

# Show only the rows where bathrooms were originally missing.
missing_rows = housing["bathrooms"].isna()
comparison = pd.DataFrame(
    {
        "size_sqft": housing.loc[missing_rows, "size_sqft"].astype(int),
        "bedrooms": housing.loc[missing_rows, "bedrooms"].astype(int),
        "original_bathrooms": housing.loc[missing_rows, "bathrooms"],
        "knn_bathrooms": imputed_housing.loc[missing_rows, "bathrooms"].round(2),
    }
)

print("scikit-learn version:", sklearn.__version__)
print("Rows imputed by nearest neighbors:")
print(comparison.to_string(index=False))



## **2. Missingness Signals**

### **2.1. Missingness Indicators**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_B/image_02_01.jpg?v=1783983468" width="250">



>* Missing values can carry useful information
>* Indicators preserve that signal after imputation

>* Missing patterns can predict important outcomes
>* Indicators distinguish imputed from observed values

>* Use indicators selectively to avoid noise
>* Validate stability and deployment relevance



In [ ]:
#@title Python Code - Missingness Indicators

# This example preserves missingness signals during imputation.
# Indicators mark values that were originally missing.
# The output compares features before and after preprocessing.

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import sklearn

# A tiny dataset keeps the missingness pattern easy to inspect.
data = pd.DataFrame(
    {
        "income_k": [55.0, np.nan, 72.0, np.nan, 48.0],
        "age": [34.0, 41.0, np.nan, 29.0, 52.0],
    }
)

# The imputer fills values and appends missingness indicator columns.
imputer = SimpleImputer(strategy="median", add_indicator=True)
filled_array = imputer.fit_transform(data)

# Indicator names show which original columns had missing values.
indicator_columns = []
for index in imputer.indicator_.features_:
    indicator_columns.append(data.columns[index] + "_was_missing")

# The transformed table contains imputed values plus missingness flags.
all_columns = list(data.columns) + indicator_columns
filled_data = pd.DataFrame(filled_array, columns=all_columns)

# A scaler can follow imputation in a normal preprocessing pipeline.
pipeline = make_pipeline(
    SimpleImputer(strategy="median", add_indicator=True),
    StandardScaler(),
)
scaled_array = pipeline.fit_transform(data)

if filled_data.shape != (5, 4):
    raise ValueError("Expected five rows and four transformed columns.")

print("scikit-learn version:", sklearn.__version__)
print("Original data with missing values:")
print(data.round(1).to_string(index=False))
print("After median imputation with indicators:")
print(filled_data.round(1).to_string(index=False))
print("Pipeline output shape after scaling:", scaled_array.shape)



### **2.2. Empty Feature Handling**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_B/image_02_02.jpg?v=1783983470" width="250">



>* Features may be entirely unobserved during fitting
>* Imputation lacks data to estimate replacements

>* Choose whether to keep or drop empty features
>* Use placeholders carefully for stable schemas

>* All-missing columns can signal data processes
>* Document and test missingness modeling choices



In [ ]:
#@title Python Code - Empty Feature Handling

# Demonstrate empty feature handling during imputation.
# Compare dropping versus keeping all-missing columns.
# Show how indicators preserve missingness signals.

import numpy as np
import pandas as pd
import sklearn
from sklearn.impute import SimpleImputer

# This training data has one entirely empty feature.
train_data = pd.DataFrame(
    {
        "age": [34.0, 45.0, np.nan, 52.0],
        "lab_score": [np.nan, np.nan, np.nan, np.nan],
        "income": [50.0, np.nan, 62.0, 70.0],
    }
)

# Future data may contain observed values in that feature.
new_data = pd.DataFrame(
    {
        "age": [40.0, np.nan],
        "lab_score": [7.5, np.nan],
        "income": [np.nan, 80.0],
    }
)

# The default imputer drops all-missing features during fitting.
default_imputer = SimpleImputer(strategy="mean")
default_train = default_imputer.fit_transform(train_data)
default_new = default_imputer.transform(new_data)

# Keeping empty features preserves the expected schema.
kept_imputer = SimpleImputer(
    strategy="constant",
    fill_value=-1,
    add_indicator=True,
    keep_empty_features=True,
)

kept_train = kept_imputer.fit_transform(train_data)
kept_new = kept_imputer.transform(new_data)
kept_names = kept_imputer.get_feature_names_out()

# Validate the key lesson before printing results.
if default_train.shape[1] >= kept_train.shape[1]:
    raise ValueError("Expected the default imputer to keep fewer columns.")

print("scikit-learn version:", sklearn.__version__)
print("Original training columns:", train_data.shape[1])
print("Default imputer columns after fit:", default_train.shape[1])
print("Keep-empty imputer columns after fit:", kept_train.shape[1])
print("Kept output names:", list(kept_names))
print("First transformed future row:", kept_new[0].round(1).tolist())



### **2.3. Native Missing Support**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_B/image_02_03.jpg?v=1783983472" width="250">



>* Models can learn directly from missing values
>* Avoids imputation that may distort patterns

>* Trees route missing values through learned splits
>* Missingness can preserve useful predictive signals

>* Validate estimator missing-value behavior carefully
>* Compare native support with imputation strategies



In [ ]:
#@title Python Code - Native Missing Support

# This example compares native missing-value support.
# Missing values can carry useful predictive signals.
# The printed scores show two modeling choices.

import numpy as np
import sklearn
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

# Create a small dataset where missingness is informative.
rng = np.random.default_rng(42)
feature = rng.normal(size=300)

# The target depends partly on whether a value is missing.
missing_signal = rng.random(300) < 0.30
target = ((feature > 0.4) | missing_signal).astype(int)

# Replace selected feature values with np.nan.
feature_with_missing = feature.copy()
feature_with_missing[missing_signal] = np.nan

# Scikit-learn expects a two-dimensional feature matrix.
X = feature_with_missing.reshape(-1, 1)
y = target

# Validate the intended missing values before modeling.
missing_count = int(np.isnan(X).sum())
if missing_count == 0:
    raise ValueError("Expected at least one missing value.")

# Split data so both models are tested fairly.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y,
)

# This model needs imputation before it can train.
imputed_model = make_pipeline(
    SimpleImputer(strategy="mean"),
    LogisticRegression(max_iter=200, random_state=42),
)

# This tree-based model accepts np.nan values directly.
native_model = HistGradientBoostingClassifier(
    max_iter=60,
    learning_rate=0.08,
    random_state=42,
)

# Fit both models on the same training split.
imputed_model.fit(X_train, y_train)
native_model.fit(X_train, y_train)

# Compare test accuracy from the two approaches.
imputed_accuracy = accuracy_score(y_test, imputed_model.predict(X_test))
native_accuracy = accuracy_score(y_test, native_model.predict(X_test))

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Missing feature values: {missing_count} of {X.size}")
print(f"Mean imputation plus logistic regression accuracy: {imputed_accuracy:.3f}")
print(f"Native missing support tree accuracy: {native_accuracy:.3f}")
print("Native support can learn from missingness without filling values first.")



## **3. Target Transformations**

### **3.1. Encoding Class Labels**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_B/image_03_01.jpg?v=1783983474" width="250">



>* Convert class names into stable numeric labels
>* Decode predictions back to original meanings

>* Encoded labels are identifiers, not measurements
>* Convert predictions back to original class names

>* Keep label mappings consistent across workflows
>* Handle unseen classes by revisiting assumptions



In [ ]:
#@title Python Code - Encoding Class Labels

# Encode text class labels for classification targets.
# LabelEncoder stores a reusable class mapping.
# Predictions can return to readable labels.

import sklearn
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier

# Small feature rows describe simple customer applications.
credit_score = [620, 710, 680, 590, 760, 640]
income_thousands = [45, 80, 62, 38, 95, 52]
features = list(zip(credit_score, income_thousands))

# The target labels are meaningful words, not numbers.
status_labels = ["denied", "approved", "approved", "denied", "approved", "denied"]
encoder = LabelEncoder()
encoded_status = encoder.fit_transform(status_labels)

# The learned mapping must be reused consistently.
print("scikit-learn version:", sklearn.__version__)
print("Classes in stored order:", list(encoder.classes_))
print("Encoded target values:", list(encoded_status))

# A classifier can train on the encoded target values.
model = DecisionTreeClassifier(max_depth=2, random_state=42)
model.fit(features, encoded_status)

# New predictions are encoded first, then decoded for people.
new_applications = [(600, 42), (735, 88)]
encoded_predictions = model.predict(new_applications)
readable_predictions = encoder.inverse_transform(encoded_predictions)

# Show both forms to connect modeling and interpretation.
print("Encoded predictions:", list(encoded_predictions))
print("Readable predictions:", list(readable_predictions))



### **3.2. Multilabel Binarization**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_B/image_03_02.jpg?v=1783983478" width="250">



>* Samples can have multiple labels simultaneously
>* Labels become binary indicator columns

>* Represent each label independently
>* Preserve column-to-label mappings carefully

>* Represent multiple outcomes with consistent label columns
>* Choose suitable models, metrics, and thresholds



In [ ]:
#@title Python Code - Multilabel Binarization

# This example converts label lists into binary targets.
# MultilabelBinarizer creates one column for each label.
# The output shows encoding and decoding clearly.

import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer

# Each sample can have several labels at once.
movie_labels = [
    ["comedy", "romance"],
    ["comedy", "drama"],
    ["action"],
    ["documentary", "history"],
]

# Fit learns the label order, then transform builds columns.
binarizer = MultiLabelBinarizer()
binary_targets = binarizer.fit_transform(movie_labels)

# A small table makes the target matrix easy to read.
target_table = pd.DataFrame(binary_targets, columns=binarizer.classes_)
print("Learned label order:", list(binarizer.classes_))
print(target_table)

# The same fitted binarizer can decode predictions later.
decoded_labels = binarizer.inverse_transform(binary_targets)
print("Decoded first movie labels:", list(decoded_labels[0]))



### **3.3. Transformed Targets**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_B/image_03_03.jpg?v=1783983476" width="250">



>* Skewed targets can make regression harder
>* Transform targets, then invert predictions

>* Log transforms model relative target changes
>* Inverse transforms keep predictions interpretable

>* Transform targets carefully to avoid distorted predictions
>* Evaluate with goal-aligned metrics and workflows



In [ ]:
#@title Python Code - Transformed Targets

# This example transforms a skewed regression target.
# The model learns log prices during training.
# Predictions return to original dollar units.

import numpy as np
import sklearn
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

# A fixed generator makes the synthetic data repeatable.
rng = np.random.default_rng(42)

# Create one feature and a positive skewed target.
home_size = rng.uniform(600, 3200, size=300)
noise = rng.normal(0, 0.28, size=300)
price_dollars = np.exp(10.8 + 0.00055 * home_size + noise)

# Scikit-learn expects a two-dimensional feature matrix.
X = home_size.reshape(-1, 1)
y = price_dollars

# Validate the target before applying a logarithm.
if np.any(y <= 0):
    raise ValueError("All target values must be positive for log1p.")

# Split data before fitting either model.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# Fit a plain linear model directly on dollar prices.
plain_model = LinearRegression()
plain_model.fit(X_train, y_train)
plain_predictions = plain_model.predict(X_test)

# Wrap the same model with a target transformation.
log_target_model = TransformedTargetRegressor(
    regressor=LinearRegression(), func=np.log1p, inverse_func=np.expm1
)
log_target_model.fit(X_train, y_train)
log_predictions = log_target_model.predict(X_test)

# Compare errors after both predictions are back in dollars.
plain_mae = mean_absolute_error(y_test, plain_predictions)
log_mae = mean_absolute_error(y_test, log_predictions)

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Plain target MAE: ${plain_mae:,.0f}")
print(f"Transformed target MAE: ${log_mae:,.0f}")
print(f"Example prediction: ${log_predictions[0]:,.0f} in original units")



# <font color="#418FDE" size="6.5" uppercase>**Imputation Targets**</font>


In this lecture, you learned to:
- Apply univariate, multivariate, and neighbor-based imputation to incomplete data. 
- Use missingness indicators and understand estimators with native missing-value support. 
- Transform classification and regression targets using scikit-learn target utilities. 

In the next Module (Module 6), we will go over 'Composite Estimators'